# Building the model by hand

This notebook builds the household water-supply MILP one constraint at a time, solves it, and
then checks — in the last cell — that it agrees with the packaged implementation in
`src/water_energy/`.

**The whole model is here.** No reduction, no sampling, no shrunken instance: 233 rows and 214
columns, small enough that the free solver bundled with `pip install gurobipy` runs it in under
a second. That was verified by *solving* under that licence, not by counting variables — some
licences are enforced at `optimize()`, so a probe that only declares variables reports success
under a licence that then refuses the real build.

You are meant to change things here. Every number the narration explains is written out in the
cell that uses it, so you can edit it and re-run. The agreement check at the bottom stays green
when you do, because both halves read the same instance data.

**If you only want to know whether the published result is correct**, you do not need this
notebook or a solver — see [`00_verification.ipynb`](00_verification.ipynb).

In [1]:
try:
    import water_energy  # noqa: F401
except ImportError:
    %pip install -q "git+https://github.com/sear-labs/water-energy-coopt-scs-2021@v1.0.0"
    import water_energy  # noqa: F401

import gurobipy as gp
from gurobipy import GRB

from water_energy import load_config, TECHS, MONTHS, TECH_NAMES
from water_energy import reference as R

print("gurobipy", gp.gurobi.version())
print("technologies:", ", ".join(f"{t} ({TECH_NAMES[t]})" for t in TECHS))

gurobipy (13, 0, 2)
technologies: RWI (rainwater, indoor), RWO (rainwater, outdoor), HGW (household greywater), CGW (community greywater), CSW (community stormwater)


## The instance data is a table, and it lives in one file

There are two kinds of number in a model like this and they are handled differently.

A **knob** is a scalar that carries a concept — a tolerance, a count, a rate. The narration
explains it, you are invited to change it, and so it is written out in the cell that uses it.

A **table** is instance data: capital costs for five technologies, rainfall for twelve months,
three price tiers. Nothing in the prose names any individual entry. If this notebook typed its
own copy and the package typed another, a failed agreement check could not tell you whether a
constraint was wrong or a digit was mistyped. So the table lives in `config.yaml` **inside the
package**, both halves read it, and the notebook hands it to the package explicitly.

`config.yaml` ships inside the wheel, which is why `pip install git+...` above is enough — a
`data/` directory at a repository root is not package data and never enters the wheel.

In [2]:
cfg = load_config()

print(f"{'technology':<24}{'capital':>10}{'operating':>11}{'energy/unit':>13}")
for t in TECHS:
    print(f"{TECH_NAMES[t]:<24}{cfg['capital_cost'][t]:>10}{cfg['operating_cost'][t]:>11}"
          f"{cfg['energy_use'][t]:>13}")

print(f"\nhouseholds        {cfg['households']}")
print(f"rainfall (mm)     {cfg['rainfall_mm']}")
print(f"price tiers ($)   {cfg['price_water']} then {cfg['price_water_tiers']}")

technology                 capital  operating  energy/unit
rainwater, indoor              692        0.0          1.1
rainwater, outdoor             508        0.0         0.55
household greywater            461        0.0          2.0
community greywater           4620        0.0          0.5
community stormwater         17323        0.0          0.1

households        100
rainfall (mm)     {1: 20, 2: 20, 3: 30, 4: 50, 5: 70, 6: 90, 7: 100, 8: 70, 9: 50, 10: 40, 11: 20, 12: 20}
price tiers ($)   5.55 then [5.97, 6.66, 8.33]


Print the dictionary form as well as the table. A frame shows you rows and columns; the model
looks values up **by key**, and every constraint below is written in terms of those keys. Seeing
`{'RWI': ..., 'RWO': ...}` makes the index set explicit rather than implied by the layout.

In [3]:
print("capital_cost  =", cfg["capital_cost"])
print("storage_cap   =", cfg["storage_cap"])
print("tier_limits   =", cfg["tier_limits"])

capital_cost  = {'RWI': 692, 'RWO': 508, 'HGW': 461, 'CSW': 17323, 'CGW': 4620}
storage_cap   = {'RWI': 3.78, 'RWO': 3.78, 'HGW': 0.09, 'CSW': 6, 'CGW': 12}
tier_limits   = [6, 4.5, 24.5]


## The knobs

Two, and both matter to what the notebook can claim.

`MIPGAP = 0.0` asks the solver to prove optimality rather than stop inside a band. The original
GAMS model shipped with `optcr = 0.05`, and at that setting the large co-optimisation model
returns a number 0.364% above its true optimum — a tolerance loose enough to be invisible in a
log, and large enough to make the answer depend on which solver you used.

`TOL` is the tolerance of the agreement check at the bottom. It is set to `1e-9`, and that is a
claim about the *computation* as much as about the two implementations: at a MIP gap of zero,
both halves prove the same optimum, so agreement far below the fourth decimal place is real
rather than lucky. If you loosen `MIPGAP` below, loosen `TOL` with it — a check that asserts
more precision than the solve delivers is testing determinism, not equivalence.

In [4]:
MIPGAP = 0.0
TOL = 1e-9

print(f"MIP gap requested : {MIPGAP}")
print(f"agreement tolerance: {TOL:.0e}")

MIP gap requested : 0.0
agreement tolerance: 1e-09


## Scaling the instance to the number of households

Everything below is per-household data multiplied out. Mechanical repetition of one step, so a
loop is honest here — there is no idea hiding in it.

One line is not mechanical: `u["CGW"] = 12.0`. The original scales every storage capacity by the
household count and then overwrites the community-greywater entry with a flat 12. It is not
derivable from anything else in the file; it is a modelling decision, transcribed.

In [5]:
h = cfg["households"]

u = {i: cfg["storage_cap"][i] * h for i in TECHS}
u["CGW"] = 12.0                                    # the override, carried across from GAMS

s0 = cfg["initial_storage"] * h
dwi = {t: cfg["demand_factor"] * cfg["indoor_demand"] * h for t in MONTHS}
dwo = {t: cfg["demand_factor"] * cfg["outdoor_demand"] * h for t in MONTHS}
rw = {t: cfg["rainfall_factor"] * cfg["rainfall_mm"][t] * h / 25.4 * (620 / 264.172)
      for t in MONTHS}
mw = [cfg["tier_limits"][k] * h for k in range(3)]

print("storage capacity   ", {k: round(v, 2) for k, v in u.items()})
print("indoor demand /mth ", round(dwi[1], 2))
print("rainwater, January ", round(rw[1], 2))
print("rainwater, July    ", round(rw[7], 2))

storage capacity    {'RWI': 378.0, 'RWO': 378.0, 'HGW': 9.0, 'CGW': 12.0, 'CSW': 600}
indoor demand /mth  576.0
rainwater, January  443.52
rainwater, July     2217.6


## The variables

Nine families. `y` is how many households adopt a technology and is continuous — the model lets
a fraction of a household adopt, which is a modelling choice inherited from the original. `y1`
is binary and answers a different question: was the technology bought *at all*.

`sf` is the only variable that may go negative, so it is the only one declaring a lower bound.
Everything else is non-negative by gurobipy's default, which matches GAMS's `Positive Variable`.

`m.update()` at the end is not decoration. gurobipy queues additions and applies them lazily, so
`m.NumVars` reads **0** until you ask for it - a progress count that is confidently wrong, and
one that shipped in the first executed copy of this notebook.

In [6]:
m = gp.Model("water_energy_by_hand")

w = m.addVars(MONTHS, name="w")                                  # utility water, tier 0
wt = {k: m.addVars(MONTHS, name=f"wt{k+1}") for k in range(3)}   # tiers 1-3
x = m.addVars(TECHS, MONTHS, name="x")                           # indoor supply
v = m.addVars(TECHS, MONTHS, name="v")                           # outdoor supply
s = m.addVars(MONTHS, name="s")                                  # storage
sf = m.addVars(MONTHS, lb=-GRB.INFINITY, name="sf")              # storage, free
e = m.addVars(MONTHS, name="e")                                  # energy used
y = m.addVars(TECHS, name="y")                                   # households adopting
y1 = m.addVars(TECHS, vtype=GRB.BINARY, name="y1")               # bought at all

m.update()          # gurobipy adds lazily; without this the counts below read 0
print(f"{m.NumVars} variables declared")

Restricted license - for non-production use only - expires 2027-11-29


214 variables declared


## The bounds — the part that is easy to lose in a port

**Grep for bounds before you believe a port is complete.** Walking the equation list is not
enough and cannot be: a bound is not an equation, and it will not appear in any list of them.

The original fixes six variables to zero among its variable *declarations* — rainwater indoors
cannot be used outdoors, community greywater is an indoor supply only, and so on. In one
comparable model, exactly one such line was missed and the objective came out 1.70 low on
72,412: far too small to notice, far too large to be arithmetic, and invisible to a check that
diffs equations.

The sign tells you what kind of bug it is. Too low on a minimisation means *under*-constrained —
something is missing, not mis-typed. That turns "search everything" into "search for a
restriction".

In [7]:
for t in MONTHS:
    v["RWI", t].UB = 0.0                    # rainwater collected indoors stays indoors
    v["CGW", t].UB = 0.0
    x["CSW", t].UB = 0.0                    # stormwater and outdoor sources stay outdoors
    x["HGW", t].UB = 0.0
    x["RWO", t].UB = 0.0

print("6 variable families fixed to zero, 12 months each")

6 variable families fixed to zero, 12 months each


## The water balance

Supply plus purchases plus what was stored last month equals demand plus what is stored now.

The first month is written separately because the original wrote it separately: `water1` and
`iwater1` are declared over all twelve months but reference month `'1'` on the right-hand side,
so GAMS generates twelve *identical* rows of each. Writing one of each is why this model has 233
rows where the original reports 270 — the feasible region is the same.

In [8]:
m.addConstr(gp.quicksum(x[i, 1] + v[i, 1] for i in TECHS)
            + w[1] + wt[0][1] + wt[1][1] + wt[2][1] + s0
            == dwi[1] + dwo[1] + sf[1], "water1")
m.addConstr(gp.quicksum(x[i, 1] for i in TECHS)
            + w[1] + wt[0][1] + wt[1][1] + wt[2][1] >= dwi[1], "iwater1")

for t in MONTHS[1:]:
    m.addConstr(gp.quicksum(x[i, t] + v[i, t] for i in TECHS)
                + w[t] + wt[0][t] + wt[1][t] + wt[2][t] + s[t - 1]
                == dwi[t] + dwo[t] + sf[t], f"water[{t}]")
    m.addConstr(gp.quicksum(x[i, t] for i in TECHS)
                + w[t] + wt[0][t] + wt[1][t] + wt[2][t] >= dwi[t], f"iwater[{t}]")

m.update()
print(f"{m.NumConstrs} constraints so far")

24 constraints so far


## The fixed-charge link, and a badly scaled big-M

> **Predict before you run the next cell.** `M` here is 9,999,999,999,999 — about 1e13 — and the
> solver's default integrality tolerance is 1e-5. A binary sitting at 1e-5 instead of 0 is
> "off" as far as the solver is concerned. How much flow does that admit through a technology
> that was never bought?

The product is about 1e8, which is four orders of magnitude larger than the entire objective.
The constraint that is supposed to force "you cannot use it unless you bought it" would, in
principle, not force it at all.

It does not bite in this instance — the binaries come back clean at exactly 1 and 0, checked
below — but it is inherited debt, and it is left alone here on purpose: tightening `M` would
change the model that the tests pin to the published answer. This is what a known, bounded
defect looks like when the right thing to do is document it rather than fix it.

In [9]:
M = cfg["big_m"]
print(f"M                        {M:,}")
print(f"integrality tolerance    {1e-5}")
print(f"flow admitted by an 'off' binary  ~{M * 1e-5:,.0f}")
print(f"the whole objective is   ~{R.GAMS_OBJECTIVE:,.0f}")

for i in TECHS:
    m.addConstr(gp.quicksum(x[i, t] for t in MONTHS) <= M * y1[i], f"investi[{i}]")
    m.addConstr(gp.quicksum(v[i, t] for t in MONTHS) <= M * y1[i], f"investo[{i}]")
    m.addConstr(y[i] <= h, f"uppery[{i}]")

m.addConstr(y["CGW"] == h * y1["CGW"], "comrestrict1")
m.addConstr(y["CSW"] == h * y1["CSW"], "comrestrict2")

m.update()
print(f"\n{m.NumConstrs} constraints so far")

M                        9,999,999,999,999
integrality tolerance    1e-05
flow admitted by an 'off' binary  ~100,000,000
the whole objective is   ~30,336

41 constraints so far


## The physical limits

Each technology can supply at most what its resource provides and at least a token amount if it
was bought at all. Twelve months of the same seven statements — mechanical, so a loop.

The pairs are worth reading as pairs. `upperrw` limits indoor rainwater by the adopting
households' share, `upperrw1` does the same outdoors, and `upperrw2` then caps the *sum* by the
total rainfall available. Without that third line the two halves could each stay inside their
own share and jointly exceed the rain that fell.

In [10]:
swi, ugw = cfg["stormwater_factor"], cfg["usable_greywater"]

for t in MONTHS:
    m.addConstr(x["RWI", t] >= 0.01 * rw[t] * y1["RWI"], f"lowerrw[{t}]")
    m.addConstr(v["RWO", t] >= 0.01 * rw[t] * y1["RWO"], f"lowerrw1[{t}]")
    m.addConstr(v["CSW", t] >= 0.01 * swi * rw[t] * y1["CSW"], f"lowersw[{t}]")
    m.addConstr(x["RWI", t] <= rw[t] / h * y["RWI"], f"upperrw[{t}]")
    m.addConstr(v["RWO", t] <= rw[t] / h * y["RWO"], f"upperrw1[{t}]")
    m.addConstr(x["RWI", t] + v["RWO", t] <= rw[t], f"upperrw2[{t}]")
    m.addConstr(v["CSW", t] <= swi * rw[t] / h * y["CSW"], f"uppersw[{t}]")
    m.addConstr(x["CGW", t] <= ugw * dwi[t] * y1["CGW"], f"upperx[{t}]")
    m.addConstr(v["HGW", t] <= ugw * dwi[t] / h * y["HGW"], f"upperx1[{t}]")
    m.addConstr(x["CGW", t] + v["HGW", t] <= ugw * dwi[t], f"upperx2[{t}]")
    m.addConstr(w[t] <= mw[0], f"upperw[{t}]")
    m.addConstr(wt[0][t] <= mw[1], f"upperwt1[{t}]")
    m.addConstr(wt[1][t] <= mw[2], f"upperwt2[{t}]")
    m.addConstr(s[t] <= gp.quicksum(u[i] * y[i] for i in TECHS), f"storagebnd[{t}]")
    m.addConstr(sf[t] == s[t], f"storage[{t}]")
    m.addConstr(gp.quicksum(cfg["energy_use"][i] * (x[i, t] + v[i, t]) for i in TECHS) == e[t],
                f"energyuse[{t}]")

m.update()
print(f"{m.NumConstrs} constraints, {m.NumVars} variables - the model is complete")

233 constraints, 214 variables - the model is complete


## The objective, in three parts

The original carried seven accounting equations — `cost`, `cost1`–`cost3`, `totale`,
`fraction1`, `fraction2` — as free variables defined by equalities. They are folded into the
expression here instead, which is why this model has 233 rows against the original's 270 without
changing the feasible region: a free variable defined by an equation constrains nothing.

That is worth remembering when reading someone else's model. **"Positive variable defined by an
equation" only constrains something if the defining expression can go negative.** In one model,
five equations that read like constraints turned out to be inert for exactly this reason.

Splitting the objective into three named parts is a presentation choice, made here because the
three answer different questions. It does not travel — for a model whose objective is one
summation over millions of terms it would only add noise.

In [11]:
cc, co = cfg["capital_cost"], cfg["operating_cost"]
pw, pwt, ps = cfg["price_water"], cfg["price_water_tiers"], cfg["pump_cost"]

z1 = (gp.quicksum(cc[i] * y[i] for i in TECHS)
      - cc["CGW"] * y["CGW"] - cc["CSW"] * y["CSW"]
      + cc["CGW"] * y1["CGW"] + cc["CSW"] * y1["CSW"])     # community tech is bought once
z2 = (gp.quicksum(co[i] * (x[i, t] + v[i, t]) for i in TECHS for t in MONTHS)
      + gp.quicksum(w[t] * pw + wt[0][t] * pwt[0] + wt[1][t] * pwt[1] + wt[2][t] * pwt[2]
                    + ps * s[t] for t in MONTHS))
z3 = gp.quicksum(cfg["price_kwh"] * e[t] for t in MONTHS)

m.setObjective(z1 + z2 + z3, GRB.MINIMIZE)
print("objective set: capital + water + energy")

objective set: capital + water + energy


## Solve

> **Predict before you run.** Community groundwater (`CGW`) has the highest capital cost of the
> five, and it is bought for all 100 households at once or not at all. Rainwater indoors (`RWI`)
> is cheap and can be adopted by a fraction of a household. Which do you expect the optimum to
> buy?

Write your answer down before running the cell. The result is in the next output.

In [12]:
m.Params.OutputFlag = 0
m.Params.MIPGap = MIPGAP
m.optimize()

assert m.Status == GRB.OPTIMAL, f"expected OPTIMAL, got status {m.Status}"

print(f"size        {m.NumConstrs} rows x {m.NumVars} cols, {m.NumBinVars} binary")
print(f"objective   {m.ObjVal:,.4f}")
print()
print(f"{'tech':<6}{'bought':>8}{'households':>13}")
for i in TECHS:
    print(f"{i:<6}{round(y1[i].X):>8}{y[i].X:>13.4f}")

size        233 rows x 214 cols, 5 binary
objective   30,336.4771

tech    bought   households
RWI          1       6.4935
RWO          1      24.4683
HGW          1       0.0000
CGW          1     100.0000
CSW          0       0.0000


Community groundwater wins, at 100 households — the fixed cost is high but it is paid once and
spread across everyone, and it is the only source big enough to displace utility water at the
higher price tiers. Rainwater indoors is bought too, but only 6.49 households' worth: it is
cheap per unit and capped by how much rain actually falls.

Community stormwater is the one technology not bought at all. Nothing about it is forbidden; it
is simply never worth its capital cost at these prices.

**Look at the `HGW` row before moving on: bought, with zero adopters.** That is not a bug, and it
is not a rounding artifact — it is a variable the optimum does not determine. Household
technologies are costed on `y`, the number of adopters, not on `y1`, the buy decision. With
`y["HGW"] = 0` the binary appears in no cost term, and both of its big-M rows have a left-hand
side of zero, so nothing in the model prefers 0 to 1.

You can check that rather than take it on trust:

```python
m.addConstr(y1["HGW"] == 0)   # then re-solve, and compare the objective
```

Forcing it each way gives `30336.477068` both times. The published GAMS run also returned 1, and
so does this one — but that is two solvers happening to pick the same vertex, not the model
saying anything. This is why the check below asserts the objective and the adoption levels
strictly, and only the binaries the optimum actually fixes.

## Did the binaries come back clean?

The big-M above admits about 1e8 of flow through an "off" technology in principle. This is where
that is checked rather than assumed: if any binary came back at 1e-6 instead of 0, the objective
would be quietly wrong and nothing else in the notebook would notice.

**A status word is not a feasibility guarantee.** `OPTIMAL` has been observed alongside a primal
residual three orders of magnitude above the solver's own promised tolerance. Read the residual.

In [13]:
worst_binary = max(min(abs(y1[i].X - 0), abs(y1[i].X - 1)) for i in TECHS)
print(f"furthest any binary sits from a clean 0/1   {worst_binary:.2e}")
print(f"reported primal residual                    {m.ConstrVio:.2e}")
print(f"largest matrix coefficient                  {M:,}")
print(f"residual, scaled by that coefficient        {m.ConstrVio / M:.2e}")

assert worst_binary < 1e-9, "a binary came back fractional - the big-M is biting"
print("\nOK - the binaries are exact, so the big-M does not bite in this instance")

furthest any binary sits from a clean 0/1   0.00e+00
reported primal residual                    3.41e-13
largest matrix coefficient                  9,999,999,999,999
residual, scaled by that coefficient        3.41e-26

OK - the binaries are exact, so the big-M does not bite in this instance


## Against the published answer

The GAMS original, re-solved with `optcr = 0` and `optca = 0` so its optimum is proven. Both
zeros are needed: `optcr` is the relative gap and `optca` the absolute one, and leaving either
at its default lets the solver stop early.

In [14]:
print(f"published (GAMS/CPLEX)   {R.GAMS_OBJECTIVE:>14,.4f}")
print(f"this notebook            {m.ObjVal:>14,.4f}")
print(f"difference               {abs(m.ObjVal - R.GAMS_OBJECTIVE):>14.2e}")

for i in TECHS:
    assert abs(y[i].X - R.GAMS_Y[i]) < R.TOL_SOLUTION, f"{i}: adoption differs"
for i in R.Y1_DETERMINED:
    assert round(y1[i].X) == R.GAMS_Y1[i], f"{i}: investment decision differs"

print("\nOK - every adoption level matches, and every binary the optimum determines")
print(f"     not asserted, because the optimum does not fix it: {', '.join(R.Y1_UNDETERMINED)}")

published (GAMS/CPLEX)      30,336.4771
this notebook               30,336.4771
difference                     3.17e-05

OK - every adoption level matches, and every binary the optimum determines
     not asserted, because the optimum does not fix it: HGW


## The agreement check

Everything above is a second implementation of a model that already exists in
`src/water_energy/model.py`. That duplication is deliberate — building it by hand *is* the
lesson — but deliberate duplication with nothing comparing the copies is just duplication with
a story attached.

So the last thing this notebook does is import the package, solve the same instance, and assert
the two agree. **Reconcile on the objective, never on the solution vector:** two correct
implementations of a degenerate model can pick different vertices, and demanding identical
variable values manufactures failures where there is no bug.

The package takes `cfg` as an argument and never re-reads the file. That is what keeps this
check green when you edit a value above and re-run — both halves see your edit.

In [15]:
from water_energy import build

packaged = build(cfg)
packaged.Params.OutputFlag = 0
packaged.Params.MIPGap = MIPGAP
packaged.optimize()

rel = abs(packaged.ObjVal - m.ObjVal) / abs(m.ObjVal)
assert rel < TOL, f"notebook and package disagree by {rel:.2e}"
print(f"notebook and package agree to {rel:.1e}")

notebook and package agree to 0.0e+00


## Now change something

The cell below is commented out on purpose. Uncomment one line, re-run the notebook from the
top, and watch two things: the answer moves, and the agreement check stays green — because the
package read the same edited `cfg` that the hand-built model did.

`scenarios/` holds two of these as data, so the ones you might reach for first need no edit at
all: `load_config("discount-10pct-payback-5yr")` and `load_config("discount-5pct-payback-10yr")`.
The original switched between them by commenting out blocks of parameters, with
`*Have to change manually` written beside them.

In [16]:
# cfg["price_kwh"] = cfg["price_kwh"] * 3          # what if electricity triples?
# cfg["capital_cost"]["CSW"] = 50                  # make stormwater cheap enough to buy
# cfg["rainfall_mm"] = {t: v * 0.5 for t, v in cfg["rainfall_mm"].items()}   # a dry year

EXPERIMENT = None
if EXPERIMENT is None:
    print("No experiment selected - uncomment a line above, or set EXPERIMENT yourself.")
    print("This notebook will not choose for you.")
else:
    raise NotImplementedError(
        "Set EXPERIMENT to a short string describing your change, and re-run from the top."
    )

No experiment selected - uncomment a line above, or set EXPERIMENT yourself.
This notebook will not choose for you.


---

## What this notebook did not do

It solved the **base household model** — the one the Python port covers. The paper's headline
results come from a full co-optimisation model of 799,394 × 900,467, which needs a commercial
solver and is not reimplemented in Python. Its source is in `model-gams/` and its published
numbers are recorded in `00_verification.ipynb`, section 4.

If you want to check the published result rather than run the implementation, that is the other
notebook, and it needs no solver at all.